# 07 — Preparar un experimento de prompting con LLM (solo train)

**Objetivo.** Generar 20 fichas de evaluación desde el conjunto original de **entrenamiento** del PSet #3, conservando el split y el hash del dataset. Se genera una muestra con 15 permanencias y 5 abandonos para mantener aproximadamente la proporción de clases.

**Control metodológico.** Nunca subir al LLM `llm_etiquetas_20_reservadas.csv`, ni revelar los valores de `CHURN_FLAG`, ni usar el hold-out. Los prompts contienen únicamente información disponible del cliente. No usar resultados de esta prueba para reoptimizar el champion ya evaluado.

La evaluación LLM es exploratoria y no es comparable con las 5 particiones de validación cruzada de los modelos tabulares.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.datos import leer_obt, variables_modelo, hash_snapshot, ORIGEN

OUT = Path('/workspace/docs')
OUT.mkdir(parents=True, exist_ok=True)
info = json.loads((OUT / 'split_info.json').read_text(encoding='utf-8'))
df = leer_obt().sort_values('CUSTOMER_ID', kind='stable').reset_index(drop=True)
assert info['origen'] == ORIGEN
assert info['hash_dataset'] == hash_snapshot(df), 'El origen cambió respecto al PSet #3'
X, y = variables_modelo(df)
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.20, random_state=int(info['seed']), stratify=y
)
idx_train, idx_test = np.sort(idx_train), np.sort(idx_test)
manifest = pd.read_csv(OUT / 'split_train_test.csv', dtype={'CUSTOMER_ID':str})
assert manifest['CUSTOMER_ID'].astype(str).tolist() == df['CUSTOMER_ID'].astype(str).tolist()
assert (manifest.iloc[idx_train]['split'] == 'train').all()
assert (manifest.iloc[idx_test]['split'] == 'test').all()
assert len(idx_train) == 5634 and len(idx_test) == 1409
print('Hash verificado:', info['hash_dataset'])
print('Train:', len(idx_train), '| Hold-out reservado:', len(idx_test))

Hash verificado: d394e39192c935f813db937647ea608341131d5b3021696fbbb186daff7043d3
Train: 5634 | Hold-out reservado: 1409


In [2]:
# Muestreo controlado en train: 15 no abandonan y 5 abandonan.
# La semilla fija hace que la muestra sea reproducible; nunca se muestrea test.
rng = np.random.default_rng(20261118)
y_entrenamiento = y.iloc[idx_train]
indices_elegidos = []
for clase, cantidad in [(0,15), (1,5)]:
    disponibles = y_entrenamiento[y_entrenamiento == clase].index.to_numpy()
    indices_elegidos.extend(rng.choice(disponibles, size=cantidad, replace=False).tolist())
rng.shuffle(indices_elegidos)
assert len(indices_elegidos) == 20
assert not set(indices_elegidos).intersection(set(idx_test))

fichas, etiquetas = [], []
for numero, indice in enumerate(indices_elegidos, start=1):
    r = X.loc[indice]
    texto = (
        'Eres un analista de churn. Con la siguiente ficha, predice únicamente '
        'ABANDONA o PERMANECE y una razón breve. No inventes datos ni atribuyas causalidad.\n'
        f"Contrato: {r['CONTRACT']}. Antigüedad: {r['TENURE_MONTHS']} meses. "
        f"Cargo mensual: {r['MONTHLY_CHARGES']}. Servicio de internet: {r['INTERNET_SERVICE']}. "
        f"Soporte técnico: {r['TECH_SUPPORT']}. Método de pago: {r['PAYMENT_METHOD']}."
    )
    fichas.append({'ejemplo':numero,'prompt':texto})
    etiquetas.append({'ejemplo':numero,'etiqueta_real_reservada':int(y.loc[indice])})

pd.DataFrame(fichas).to_csv(OUT/'llm_prompts_20_sin_etiquetas.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(etiquetas).to_csv(OUT/'llm_etiquetas_20_reservadas.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'ejemplo':range(1,21),'prediccion_llm':['']*20,'razon_llm':['']*20}).to_csv(
    OUT/'llm_respuestas_20.csv', index=False, encoding='utf-8-sig'
)
print('Generados 20 prompts de TRAIN y etiquetas reservadas por separado.')
print('ENVIAR AL LLM SOLO: docs/llm_prompts_20_sin_etiquetas.csv')
print('NO ENVIAR: docs/llm_etiquetas_20_reservadas.csv')

Generados 20 prompts de TRAIN y etiquetas reservadas por separado.
ENVIAR AL LLM SOLO: docs/llm_prompts_20_sin_etiquetas.csv
NO ENVIAR: docs/llm_etiquetas_20_reservadas.csv


## Siguiente operación

Sube al chat únicamente **`docs/llm_prompts_20_sin_etiquetas.csv`** (sin respuestas ni etiquetas). Solicita una clasificación individual para cada `ejemplo` y pide el archivo `llm_respuestas_20.csv` con columnas `ejemplo`, `prediccion_llm`, `razon_llm`.

Cuando lo completes, **no vuelvas a muestrear**. Ejecuta el notebook `08_llm_evaluar.ipynb` para calcular resultados contra las etiquetas locales. Registra el modelo, la fecha, el prompt y si se trabajó manualmente o por API.